In [ ]:
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, countDistinct, year, month

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!ls "/content/drive/MyDrive"

In [ ]:
spark = SparkSession.builder \
    .appName("Amazon Reviews EDA") \
    .config("spark.sql.files.maxPartitionBytes", "64m") \
    .config("spark.sql.shuffle.partitions", "128") \
    .getOrCreate()

In [ ]:
reviews_path = "/content/drive/MyDrive/Amazon Project - Draft 01/outputs/reviews_eda_sample.parquet"
df_reviews = spark.read.parquet(reviews_path)
items_path = "/content/drive/MyDrive/Amazon Project - Draft 01/outputs/meta.parquet"
df_items = spark.read.parquet(items_path)

In [ ]:
df_items.select("main_category").distinct().show()

In [ ]:
df_reviews.head()

In [ ]:
df_items.head()

In [ ]:
df_reviews = (
    df_reviews
    .withColumnRenamed("parent_asin", "item_id")
    .withColumnRenamed("images", "review_images")
)

df_items = df_items.withColumnRenamed("parent_asin", "item_id")

In [ ]:
df = df_reviews.join(df_items, on="item_id", how="left")

In [ ]:
from pyspark.sql.functions import array_contains, lower
df = df.filter(
    (lower(col("main_category")) == "electronics") |
    (array_contains(col("categories"), "Electronics"))
)

In [ ]:
df.head()

In [ ]:
# Step 1: Install required libraries
!pip install transformers sentence-transformers

# Step 2: Import libraries
from sentence_transformers import SentenceTransformer
import pandas as pd

# Step 3: Sample 1000 review texts from Spark DataFrame
pdf = df.select("text").limit(1000).toPandas()
texts = pdf["text"].fillna("").tolist()

# Step 4: Load a pre-trained BERT model optimized for sentence embeddings
model = SentenceTransformer("all-MiniLM-L6-v2")

# Step 5: Encode each review into a semantic vector
embeddings = model.encode(texts, show_progress_bar=True)

In [ ]:
# Step 6: Convert embeddings to DataFrame for inspection or downstream tasks
import numpy as np
embedding_df = pd.DataFrame(embeddings)
embedding_df["original_text"] = texts

# Step 7: Preview the result
embedding_df.head()

In [ ]:
# Step 6: Convert embeddings to DataFrame
import numpy as np
embedding_df = pd.DataFrame(embeddings)
embedding_df["original_text"] = texts

# Step 7: Add rating from original Spark DataFrame
pdf = df.select("text", "rating").limit(4000).toPandas()
embedding_df["rating"] = pdf["rating"]

# Step 8: Add 3-class sentiment label using TextBlob
from textblob import TextBlob

def get_sentiment(text):
    polarity = TextBlob(text).sentiment.polarity
    if polarity > 0.1:
        return "positive"
    elif polarity < -0.1:
        return "negative"
    else:
        return "neutral"

embedding_df["sentiment"] = embedding_df["original_text"].apply(get_sentiment)

# Step 9: Apply t-SNE using only numeric embedding columns
from sklearn.manifold import TSNE

# Automatically select only numeric columns (i.e., BERT embeddings)
embedding_columns = embedding_df.select_dtypes(include=[np.number]).columns.tolist()

# Apply t-SNE
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
embeddings_2d = tsne.fit_transform(embedding_df[embedding_columns].values)

# Add t-SNE results to DataFrame
embedding_df["x"] = embeddings_2d[:, 0]
embedding_df["y"] = embeddings_2d[:, 1]

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines

# Step 1: 定义颜色映射（情感）
color_map = {"positive": "green", "neutral": "gray", "negative": "red"}
embedding_df["color"] = embedding_df["sentiment"].map(color_map)

# Step 2: 定义评分对应的形状（marker）
def get_marker(rating):
    if rating == 1:
        return "o"  # circle
    elif rating == 2:
        return "x"  # cross
    elif rating == 3:
        return "s"  # square
    elif rating == 4:
        return "D"  # diamond
    elif rating == 5:
        return "^"  # triangle
    else:
        return "."  # fallback

embedding_df["marker"] = embedding_df["rating"].apply(get_marker)

# Step 3: 绘制图表
plt.figure(figsize=(10, 6))

# 遍历所有组合：情感 × 评分形状
unique_markers = ["o", "x", "s", "D", "^"]
for sentiment in color_map:
    for marker in unique_markers:
        subset = embedding_df[
            (embedding_df["sentiment"] == sentiment) &
            (embedding_df["marker"] == marker)
        ]
        if not subset.empty:
            plt.scatter(
                subset["x"],
                subset["y"],
                c=color_map[sentiment],
                marker=marker,
                label=f"{sentiment} + rating {marker}",
                alpha=0.7,
                edgecolors="w"
            )

# Step 4: 自定义图例（颜色 + 形状）
legend_elements = [
    mpatches.Patch(color="green", label="Positive"),
    mpatches.Patch(color="gray", label="Neutral"),
    mpatches.Patch(color="red", label="Negative"),
    mlines.Line2D([], [], color="black", marker="o", linestyle="None", markersize=8, label="Rating = 1"),
    mlines.Line2D([], [], color="black", marker="x", linestyle="None", markersize=8, label="Rating = 2"),
    mlines.Line2D([], [], color="black", marker="s", linestyle="None", markersize=8, label="Rating = 3"),
    mlines.Line2D([], [], color="black", marker="D", linestyle="None", markersize=8, label="Rating = 4"),
    mlines.Line2D([], [], color="black", marker="^", linestyle="None", markersize=8, label="Rating = 5"),
]

plt.legend(handles=legend_elements, title="Sentiment & Rating", loc="best")
plt.title("Semantic Distribution by Sentiment (Color) and Rating (Shape)")
plt.xlabel("t-SNE Component 1")
plt.ylabel("t-SNE Component 2")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
df.printSchema()

In [ ]:
from pyspark.sql.functions import size
import matplotlib.pyplot as plt

# Step 1: 统计每条评论的图片数量
df_img_count = df.withColumn("num_images", size("review_images"))

# Step 2: 按图片数量分组统计评论数量
img_stats = df_img_count.groupBy("num_images").count().orderBy("num_images")

# Step 3: 转换为 Pandas DataFrame
img_stats_pd = img_stats.toPandas()

# Step 4: 绘制柱状图
plt.figure(figsize=(8, 5))
plt.bar(img_stats_pd["num_images"], img_stats_pd["count"], color="skyblue", edgecolor="black")

# Step 5: 添加图表元素
plt.xlabel("Number of Images per Review")
plt.ylabel("Number of Reviews")
plt.title("Distribution of Review Image Counts")
plt.xticks(img_stats_pd["num_images"])
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
from pyspark.sql.functions import size, avg
import matplotlib.pyplot as plt
import seaborn as sns

# Step 1: add images number column
df_with_count = df.withColumn("num_images", size("review_images"))

# Step 2: group by item_id and compute the average images and average rating
agg_df = df_with_count.groupBy("item_id").agg(
    avg("num_images").alias("avg_images"),
    avg("rating").alias("avg_rating")
)

# Step 3: transfer to Pandas DataFrame
pdf = agg_df.toPandas()

In [ ]:
from pyspark.sql.functions import size, avg
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

df_with_count = df.withColumn("num_images", size("review_images"))

agg_df = df_with_count.groupBy("item_id").agg(
    avg("num_images").alias("avg_images"),
    avg("rating").alias("avg_rating")
)

pdf = agg_df.toPandas()

# Step 4: set avg_images bins
bins = [0, 0.5, 1, 2, 3, 5, 10]
labels = ["0", "0.5", "1–2", "2–3", "3–5", "5+"]
pdf["image_bin"] = pd.cut(pdf["avg_images"], bins=bins, labels=labels, include_lowest=True)

# Step 5: Visualization
plt.figure(figsize=(10,6))
sns.violinplot(x="image_bin", y="avg_rating", data=pdf, palette="Blues")
plt.title("Rating Distribution by Avg Review Image Count (Binned)")
plt.xlabel("Average Number of Images per Review (Binned)")
plt.ylabel("Average Rating")
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
pip install pandas scikit-learn wordcloud matplotlib

In [ ]:
pdf = df.select("text").limit(4000).toPandas()

In [ ]:

pdf = df.select("text").limit(4000).toPandas()
texts = pdf["text"].dropna().tolist()

# pip install pandas scikit-learn wordcloud matplotlib

pdf = df.select("text").limit(4000).toPandas()
texts = pdf["text"].dropna().tolist()

from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

custom_stopwords = {
    "use", "just", "like", "dont", "little", "br", "very", "its",
    "you", "have", "not", "are", "bought", "product", "really", "using", "need", "doe"
}

stop_words = list(set(ENGLISH_STOP_WORDS).union(custom_stopwords))

from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    stop_words=stop_words,
    ngram_range=(1, 3),
    max_features=1000
)

tfidf_matrix = vectorizer.fit_transform(texts)

feature_names = vectorizer.get_feature_names_out()
tfidf_scores = tfidf_matrix.sum(axis=0).A1
tfidf_dict = dict(zip(feature_names, tfidf_scores))

top_tfidf = dict(sorted(tfidf_dict.items(), key=lambda x: x[1], reverse=True)[:100])

from wordcloud import WordCloud
import matplotlib.pyplot as plt

wc = WordCloud(
    width=1000,
    height=500,
    background_color="white",
    colormap="viridis"
)
wc.generate_from_frequencies(top_tfidf)

plt.figure(figsize=(14, 7))
plt.imshow(wc, interpolation="bilinear")
plt.axis("off")
plt.title("Top Keywords from Reviews (TF-IDF + N-gram)", fontsize=16)
plt.tight_layout()
plt.show()